# Adding Conversions to the Energy System Model

In the [previous notebook](../_01_initialize/_1_initialize_ESM.ipynb), we initialized an energy system model, which defines the basic structure of the energy system such as locations, commodities, and the temporal resolution.

In this notebook, we introduce **Conversions components**. Conversions components represent components that **converts one or several commodities into other commodities of the Energy System  Model**. They can be seen as black boxes, using the inputs commodities to create the output commodities. We focus in this notebook on the most essential parameters required to define and understand a Conversion component, while more advanced and optional settings will be explained in subsequent notebooks.

Typical examples of sources include:

- a co-generation power plant using natural gas to produce electricity and heat 
- an electrolyzer using electricity to produce hydrogen
- a chemical plant using hydrogen, CO2, heat and electricity to produce methanol


## Add Conversions

## Load ESM

We first load the ESM from the [previous notebook](../_02_add_component/_2_add_sink.ipynb).

In [2]:
import fine as fn
import fine.IOManagement.xarrayIO as xrIO
import pandas as pd
import numpy as np
from pathlib import Path
cwd = Path.cwd().resolve()
base_path = cwd.parents[2]
nc_file = base_path / "examples" / "Examples" / "NetCDF" / "esm_source_sink_transmission.nc"

esM = xrIO.readNetCDFtoEnergySystemModel(nc_file)

### Combined Cycle gas turbine plant

We can now add combined cycle gas turbine plant as a conversion component. Below you can find a more detailed explanation of the parameters used here.

In [2]:
esM.add(
    fn.Conversion(
        esM=esM,
        name="CCGT plants (methane)",
        physicalUnit=r"GW$_{el}$",
        commodityConversionFactors={
            "electricity": 1,
            "naturalGas": -1 / 0.6,
            "CO2": 201 * 1e-6 / 0.6,
        },
        hasCapacityVariable=True,
        investPerCapacity=0.65,
        opexPerCapacity=0.021,
        interestRate=0.08,
        economicLifetime=33,
    )
)

## Save the Energy System Model

In [3]:
cwd = Path.cwd().resolve()
base_path = cwd.parents[2]
nc_file = base_path / "examples" / "Examples" / "NetCDF" / "esm_source_sink_transmission_conversion.nc"

xrIO.writeEnergySystemModelToNetCDF(
    esM, outputFilePath=nc_file, overwriteExisting=True
)


Writing output to netCDF... 
Done. (0.4555 sec)


## General Structure of a Conversion Instance

The following code snippet shows the structure of the `Conversion` class and its arguments.

```python
Conversion(
    esM,                                    # defined in this notebook
    name,                                   # defined in this notebook
    physicalUnit,                           # defined in this notebook
    commodityConversionFactors,             # defined in this notebook
    hasCapacityVariable=True,               # defined in this notebook
    capacityVariableDomain="continuous",
    capacityPerPlantUnit=1,
    linkedConversionCapacityID=None,
    hasIsBuiltBinaryVariable=False,
    bigM=None,
    operationRateMin=None,                  # defined in this notebook
    operationRateMax=None,                  # defined in this notebook
    operationRateFix=None,                  # defined in this notebook
    tsaWeight=1,
    locationalEligibility=None,
    capacityMin=None,
    capacityMax=None,
    partLoadMin=None,
    sharedPotentialID=None,
    linkedQuantityID=None,
    capacityFix=None,
    commissioningMin=None,
    commissioningMax=None,
    commissioningFix=None,
    isBuiltFix=None,
    investPerCapacity=0,
    investIfBuilt=0,
    opexPerOperation=0,
    opexPerCapacity=0,
    opexIfBuilt=0,
    QPcostScale=0,
    interestRate=0.08,
    economicLifetime=10,
    technicalLifetime=None,
    yearlyFullLoadHoursMin=None,
    yearlyFullLoadHoursMax=None,
    stockCommissioning=None,
    floorTechnicalLifetime=True,
    commissioningDependentCcf=False,
    emissionFactors=None,
    flowShares=None,
    pwlcfParameters=None,
    rampUpMax=None,
    rampDownMax=None,
    useTemporalCyclicConstraints=True,
)
```
In the following sections, we explain the most important arguments of a Conversion component.


## Required Arguments

### esM

`esM` is the energy system model to which the conversion is added.

### name

`name` is a string, which should describe the type of conversion which is added to the energy system model.

Examples:
- "nat_gas_power_plant"
- "PEM_electrolyzer"

### physicalUnit

`physicalUnit` defines the reference physical unit of the conversion component, to which maximum capacity limitations, cost parameters and the operation time series are all expressed.

Examples: 
- if `physicalUnit` = MW_{H2} for an electrolyzer, it means that its `capacityMax` of 10 MW is referred to hydrogen capacity and not electricity. 

### commodityConversionFactors

`commodityconversionfactor` specifies the conversion factors with which commodities are converted into each other with one unit of operation. The unit of operation were defined ealier in the [EnergySystemModel Initialization](../_01_initialize/_1_initialize_ESM.ipynb). The conversion factor related to this commodity is given as a float (constant), pandas.Series or pandas.DataFrame (time-variable). A negative value indicates that the commodity is consumed. A positive value indicates that the commodity is produced. Check unit consistency when specifying this parameter!

Examples: 
An electrolyzer converts, simply put, electricity into hydrogen with an electrical efficiency of 70%. The physicalUnit is given as GW_electric, the unit for the 'electricity' commodity is given in GW_electric and the 'hydrogen' commodity is given in GW_hydrogen_lowerHeatingValue -> the commodityConversionFactors are defined as {'electricity':-1,'hydrogen':0.7}.


## Optional Parameters

### hasCapacityVariable

`hasCapacityVariable` was alreadz defined previouslz, but you can [see it here!](_2_add_sink.ipynb#economicLifetime)

### operationRateMax

`operationRateMax` was already defined previously, but you can [see it here!](_2_add_sink.ipynb#economicLifetime)

### capacityMax 
`capacityMax` was already defined previously, but you can [see it here!](_2_add_sink.ipynb#economicLifetime)

### investPerCapacity

`investPerCapacity` was already defined previously, but you can [see it here!](_2_add_sink.ipynb#economicLifetime)

### opexPerCapacity

`opexPerCapacity` was already defined previously, but you can [see it here!](_2_add_sink.ipynb#economicLifetime)


### opexPerOperation

`opexPerOperation` describes the cost for one unit of the operation. The cost which is directly proportional to the operation of the component is obtained by multiplying the opexPerOperation parameter with the annual sum of the operational time series of the components. The opexPerOperation can either be given as a float or a Pandas Series with location specific values. The cost unit in which the parameter is given has to match the one specified in the energy system model (e.g. Euro, Dollar, 1e6 Euro). |br| * the default value is 0 :type opexPerOperation: * Pandas Series with positive (>=0) entries. The indices of the series have to equal the in the energy system model specified locations. * a dictionary with investment periods as keys and one of the two options above as values.

### interestRate

`interestRate` was already defined previously, but you can [see it here!](_2_add_sink.ipynb#economicLifetime)

### economicLifetime

`economicLifetime` was already defined previously, but you can [see it here!](_2_add_sink.ipynb#economicLifetime)

Many parameters were left out here. Some of them might need a page on their own. Others could be collected in an "other features" notebook

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "NetCDF"))
from docstringTable import display_param_table


display_param_table(fn.conversion)

NameError: name 'fn' is not defined